# AI GRC Lab
## Applying the NIST AI Risk Management Framework
##  Bias Risk Assessment: Facial Recognition
### Run the full lab here: https://colab.research.google.com/drive/1hfLfG_XoKMKRJyWz_kiFNkHzHI8TFgV0?usp=sharing

**Author:** Jake Kantor, Greg Arcand, Marcus Lee
**Institution:** Old Dominion University — M.S. Cybersecurity, AI Security  
**Course:** CYSE 640: Trustworthy and Responsible AI  
**Framework:** NIST AI RMF 1.0  
**Dataset:** UTKFace (23,708 facial images)

## Setup
This notebook requires the UTKFace dataset.  
Download from: https://susanqq.github.io/UTKFace/  
Upload to Google Drive and update DATA_DIR in Part A to your path before running.

## Part A : Data Loading and Initial Exploration

In [ ]:
!pip install -q lime diffusers transformers accelerate diffprivlib

In [ ]:
print("Starting imports")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
from torchvision import transforms

import tensorflow as tf
from tensorflow.keras import layers, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    classification_report
)
from lime import lime_image
from skimage.segmentation import mark_boundaries

from datasets import Dataset
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)
import random

from diffusers import DiffusionPipeline

from diffprivlib.models import LogisticRegression as DPLogisticRegression
from sklearn.linear_model import LogisticRegression

from google.colab import drive

# Check if drive is already mounted
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# For TensorFlow
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        # Currently, memory growth needs to be the same across GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
    except RuntimeError as e:
        # Memory growth must be set before GPUs have been initialized
        print(e)
else:
    print("No GPU found, running on CPU.")

# For PyTorch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

print("Imports done")

### Load UTKFace Data

The UTKFace dataset contains 23,708 facial images labeled with age, gender, and race.
This assessment focuses on binary gender classification (0 = Female, 1 = Male).

Update the DATA_DIR path below to your local UTKFace dataset location before running.

## Dataset Description

**Source:** UTKFace — large publicly available facial image dataset for AI research  
**Official URL:** https://susanqq.github.io/UTKFace/

**Filename Format:** `AGE_GENDER_RACE_DATE.jpg`

| Label | Values |
|-------|--------|
| Gender | 0 = Female, 1 = Male |
| Race | 0 = White, 1 = Black, 2 = Asian, 3 = Indian, 4 = Other |

**Size:** 23,708 facial images  
**Prediction Task:** Binary gender classification (Female / Male)

In [ ]:
# Load UTKFace Data

# Update this path to your UTKFace dataset location before running
DATA_DIR = "/content/drive/MyDrive/CYSE640/Midterm/UTKFace"

image_files = os.listdir(DATA_DIR)

print("Total images:", len(image_files))
print("Example filenames:", image_files[:5])
print("image_files[0] =", image_files[0])

data = []
for file in image_files:
    try:
        age, gender, race, _ = file.split("_",3)
        data.append({
            "filename":file,
            "age":int(age),
            "gender":int(gender),
            "race":int(race)
        })
    except:
        continue

print("data[0] =", data[0])

df = pd.DataFrame(data)

print(df.head())
print(df.describe())

### Preprocessing & Label Setup

Images are resized to 64×64 pixels and converted to grayscale tensors.
Gender labels are extracted from filenames (0 = Female, 1 = Male).
Dataset is split 80/20 for training and testing with stratification by gender.

In [ ]:
# Convert Images to 64x64

transform = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.ToTensor()
])

X = []
y = []
races = []

print("Transforms starting, takes awhile. Limit x for speed")

for i, row in df.iterrows():

    img_path = os.path.join(DATA_DIR, row["filename"])

    try:
        image = Image.open(img_path).convert("L")
        image = transform(image)

        X.append(image)
        y.append(row["gender"])
        races.append(row["race"])
    except Exception as e:
        print(f"Error loading image {row['filename']}: {e}")
        continue

    if i == 10:
        print("i =", i)
    if i == 100:
      print("i =", i)
    if i == 500:
      print("i =", i)
    if i == 750:
      print("i =", i)

    if len(X) >= 1000:
        break

# Extract features:
# Transform images into tensors

X = torch.stack(X)
y = torch.tensor(y)
races = np.array(races)

print("Images loaded:", len(X))

# Convert to TensorFlow-compatible format
X_tf = tf.convert_to_tensor(X.permute(0, 2, 3, 1).numpy(), dtype=tf.float32)

print("Tensor shape:", X_tf.shape)

### Data Exploration

Summary statistics for gender and race distribution across the full 23,708 image dataset.

In [ ]:
# Print Summary Stats
print("Gender Distribution:")
print(df["gender"].value_counts())
print(df["gender"].describe())

print("\nRace Distribution:")
print(df["race"].value_counts())
print(df["race"].describe())

## Part B: Baseline Model + Robustness & Resilience

### Baseline Model

A simple CNN architecture trained for 4 epochs on the 800-image training subset.
Architecture: Conv2D(32) → MaxPool → Conv2D(64) → MaxPool → Flatten → Dense(64) → Dense(1, sigmoid)

**Note on Device Usage:** TensorFlow models in Parts B through F run on CPU to avoid conflicts with PyTorch GPU allocation. Part G uses GPU for generative model inference.

In [ ]:
# Train a Baseline Model on CPU

# Split data (include race for fairness check)
X_train_np, X_test_np, y_train_np, y_test_np, race_train, race_test = train_test_split(
    X_tf.numpy(),
    y.numpy(),
    races,
    test_size = 0.2,
    stratify = y.numpy(),
    random_state = 42
)

# Convert to TF tensors
X_train = tf.convert_to_tensor(X_train_np, dtype=tf.float32)
y_train = tf.convert_to_tensor(y_train_np, dtype=tf.float32)
X_test = tf.convert_to_tensor(X_test_np, dtype=tf.float32)
y_test = tf.convert_to_tensor(y_test_np, dtype=tf.float32)

print("X_train shape:", X_train.shape)

# Use CPU to avoid GPU conflicts with PyTorch later
with tf.device('/CPU:0'):
    def build_model(input_dim):
        model = models.Sequential([
            layers.Input(shape=input_dim),
            layers.Conv2D(32,(3,3),activation='relu'),
            layers.MaxPooling2D((2,2)),
            layers.Conv2D(64,(3,3),activation='relu'),
            layers.MaxPooling2D((2,2)),
            layers.Flatten(),
            layers.Dense(64,activation='relu'),
            layers.Dense(1,activation='sigmoid')
        ])
        return model

    def compile_model(model):
        model.compile(
            optimizer="adam",
            loss="binary_crossentropy",
            metrics=["accuracy"]
        )

    baseline = build_model(X_train.shape[1:])
    compile_model(baseline)
    baseline.summary()

    print("Starting training on CPU...")
    history = baseline.fit(X_train, y_train, epochs=4, validation_data=(X_test, y_test))
    print("Training complete.")

In [ ]:
# Train a Baseline Model
with tf.device('/CPU:0'):
  X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
      X_tf.numpy(), # Convert X_tf to a NumPy array
      y.numpy(), # Convert y to a NumPy array
      test_size = 0.2,
      stratify = y.numpy(), # Convert y to a NumPy array for stratify as well
      random_state = 42
  )

  # Explicitly convert to TensorFlow tensors for training
  X_train = tf.convert_to_tensor(X_train_np, dtype=tf.float32)
  y_train = tf.convert_to_tensor(y_train_np, dtype=tf.float32)
  X_test = tf.convert_to_tensor(X_test_np, dtype=tf.float32)
  y_test = tf.convert_to_tensor(y_test_np, dtype=tf.float32)

  print("For reference.\n")
  print("X_train shape:", X_train.shape)

  baseline = build_model(X_train.shape[1:])

  compile_model(baseline)

  # Print model summary; train for a few epochs on your training subset.

  baseline.summary()

  baseline.fit(X_train, y_train, epochs=4, validation_data=(X_test, y_test))

### Initial Evaluation

Model evaluated on 200-image test set. Results include accuracy, confusion matrix, and classification report with per-class precision, recall, and F1 scores.

In [ ]:
with tf.device('/CPU:0'):
  # Evaluate a test subset or validation approach.

  y_pred = (baseline.predict(X_test) > 0.5).astype(int)

  # Show accuracy, confusion matrix, and classification report.

  accuracy = accuracy_score(y_test, y_pred)

  print("Accuracy:",accuracy)

  print("\nConfusion Matrix")
  print(confusion_matrix(y_test, y_pred))

  print("\nClassification Report")
  print(classification_report(y_test, y_pred))

  # Stress Test [e.g., add random noise or slight image perturbations
  # (rotation, brightness changes) to a small sample].

  noise = np.random.normal(.2, 0.2, X_test.shape)

  X_test_noisy = np.clip(X_test + noise, 0, 1)

  # Check model performance before vs. after perturbation.
  loss, acc_noisy = baseline.evaluate(X_test_noisy, y_test)
  print("Accuracy with noise:", acc_noisy)

#### Risk Analysis

Facial recognition models risk systematic misclassification across demographic groups due to training data imbalance. False positives and false negatives distributed unevenly across race and gender categories create discriminatory outcomes in high-stakes deployment contexts including hiring, surveillance, and access control — precisely the use cases regulated under the EU AI Act and NYC Local Law 144.

### Robustness Testing

Gaussian noise perturbation applied to test images to evaluate model stability under real-world image quality variation. Accuracy compared before and after perturbation to quantify degradation.

In [ ]:
with tf.device('/CPU:0'):
  # Evaluate a test subset or validation approach.
  y_pred = (baseline.predict(X_test) > 0.5).astype(int)

  # Show accuracy, confusion matrix, and classification report.

  accuracy = accuracy_score(y_test, y_pred)

  print("Accuracy:",accuracy)

  print("\nConfusion Matrix")
  print(confusion_matrix(y_test, y_pred))

  print("\nClassification Report")
  print(classification_report(y_test, y_pred))

  noise = np.random.normal(.2, 0.2, X_test.shape)

  X_test_noisy = np.clip(X_test + noise, 0, 1)

  # Check model performance before vs. after perturbation.
  loss, acc_noisy = baseline.evaluate(X_test_noisy, y_test)
  print(f"Accuracy with noise: {acc_noisy: .4f}")

### Robustness Analysis

Gaussian noise degraded accuracy from 70% to 46.5%, confirming sensitivity to real-world image quality variation. More critically, the FGSM adversarial attack reduced accuracy to 14.5% — a 55.5 percentage point drop from a perturbation invisible to the human eye. This extreme vulnerability has direct governance implications for any deployment context where adversarial manipulation is possible.

**Recommended mitigations:**
- **Adversarial training** — demonstrated in Part C to achieve 100% accuracy against FGSM attacks, with a modest clean accuracy tradeoff (57.5% vs 70%)
- **Data augmentation** — rotations, brightness changes, and flips improve generalization to natural image variation
- **Robust architectures** — regularization and ensemble methods provide additional stability

## Part C: Adversarial Testing


### FGSM Adversarial Attack

Fast Gradient Sign Method (FGSM) attack applied to test images at epsilon=0.05 (subset) and epsilon=0.1 (full test set). Clean accuracy compared against adversarial accuracy to quantify model vulnerability.

In [ ]:
with tf.device('/CPU:0'):
    print("\n--- Adversarial Testing ---")

    def fgsm_attack(model, X, y, epsilon=0.1):
        # Ensure inputs are tensors
        X = tf.convert_to_tensor(X, dtype=tf.float32)
        y = tf.convert_to_tensor(y, dtype=tf.float32)

        # Reshape y to match predictions shape (N, 1)
        y = tf.reshape(y, (-1, 1))

        with tf.GradientTape() as tape:
            tape.watch(X)
            predictions = model(X)
            # Binary Crossentropy for binary classification
            loss = tf.keras.losses.binary_crossentropy(y, predictions)

        gradient = tape.gradient(loss, X)
        signed_grad = tf.sign(gradient)
        adversarial_X = X + epsilon * signed_grad
        adversarial_X = tf.clip_by_value(adversarial_X, 0, 1)
        return adversarial_X

    # Generate adversarial examples on a subset of test data for speed
    X_test_subset = X_test[:100]
    y_test_subset = y_test[:100]

    epsilon = 0.05
    print(f"Generating adversarial examples (epsilon={epsilon})...")
    # Use the baseline model from the previous step
    X_adv = fgsm_attack(baseline, X_test_subset, y_test_subset, epsilon)

    # Evaluate
    loss_clean, acc_clean = baseline.evaluate(X_test_subset, y_test_subset, verbose=0)
    loss_adv, acc_adv = baseline.evaluate(X_adv, y_test_subset, verbose=0)

    print(f"Accuracy on Clean Images: {acc_clean:.4f}")
    print(f"Accuracy on Adversarial Images: {acc_adv:.4f}")

    # Visualize Adversarial Examples
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.imshow(X_test_subset[0].numpy().squeeze(), cmap='gray')
    plt.title(f"Original (True: {int(y_test_subset[0])})")
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(X_adv[0].numpy().squeeze(), cmap='gray')
    plt.title("Adversarial")
    plt.axis('off')

    plt.subplot(1, 3, 3)
    # Show absolute difference to highlight perturbation
    plt.imshow(np.abs(X_test_subset[0].numpy() - X_adv[0].numpy()).squeeze(), cmap='hot')
    plt.title("Perturbation")
    plt.axis('off')
    plt.show()

In [ ]:
with tf.device('/CPU:0'):
  # Generate advesarial training data
  X_train_adv = fgsm_attack(baseline, X_train, y_train)
  X_train_mix = np.vstack((X_train, X_train_adv))
  y_train_mix = np.hstack((y_train, y_train))

  idx = np.random.permutation(len(X_train_mix))
  X_train_mix, y_train_mix = X_train_mix[idx], y_train_mix[idx]

  # Build and train advesarial model on training set
  adv_trained = build_model(X_train_adv.shape[1:])

  compile_model(adv_trained)

  adv_trained.summary()

  adv_trained.fit(X_train_mix, y_train_mix, epochs=4, validation_data=(X_test, y_test))

In [ ]:
with tf.device('/CPU:0'):
  # Generate adversarial examples for the test set using the baseline model
  print("Generating adversarial test examples...")
  X_test_adv = fgsm_attack(baseline, X_test, y_test)

  # Verification for X_test_adv
  print(f"\n=== X_test_adv Properties ===")
  print(f"Shape: {X_test_adv.shape}")
  print(f"Data Type: {X_test_adv.dtype}")
  print(f"Min Value: {tf.reduce_min(X_test_adv):.4f}")
  print(f"Max Value: {tf.reduce_max(X_test_adv):.4f}")

  # Calculate mean absolute difference
  mean_abs_diff = tf.reduce_mean(tf.abs(X_test.numpy() - X_test_adv)).numpy()
  print(f"Mean Absolute Difference (X_test vs X_test_adv): {mean_abs_diff:.4f}")
  print(f"================================")

  borders = "=" * 15
  print(f"{borders} Evaluation results {borders}")

  # Evaluate the baseline model on adversarial test examples
  loss_baseline_adv, acc_baseline_adv = baseline.evaluate(X_test_adv, y_test, verbose=0)
  print(f"Baseline Model Accuracy on Adversarial Test Set: {acc_baseline_adv:.4f}")

  # Evaluate the adversarially trained model on clean test examples
  loss_adv_trained_clean, acc_adv_trained_clean = adv_trained.evaluate(X_test, y_test, verbose=0)
  print(f"Adversarially Trained Model Accuracy on Clean Test Set: {acc_adv_trained_clean:.4f}")

  # Evaluate the adversarially trained model on adversarial test examples
  loss_adv_trained_adv, acc_adv_trained_adv = adv_trained.evaluate(X_test_adv, y_test, verbose=0)
  print(f"Adversarially Trained Model Accuracy on Adversarial Test Set: {acc_adv_trained_adv:.4f}")


### Adversarial Training Results

The adversarially trained model achieved 100% accuracy on FGSM adversarial examples because training on a mixed dataset of clean and perturbed images explicitly taught the model to classify through epsilon=0.1 perturbations. However, this robustness is attack-specific — performance against PGD, Carlini-Wagner, or FGSM at different epsilon values remains untested.

**Key findings:**

| Condition | Accuracy |
|-----------|----------|
| Baseline — clean images | 70.0% |
| Baseline — adversarial images (ε=0.1) | 14.5% |
| Adversarially trained — clean images | 57.5% |
| Adversarially trained — adversarial images (ε=0.1) | 100.0% |

Mean absolute perturbation magnitude: 0.0927 (confirming imperceptible pixel-level changes produce catastrophic accuracy degradation in the undefended model)

**Robustness-utility tradeoff:** Adversarial training improved adversarial robustness by 85.5 percentage points while reducing clean accuracy by 12.5 percentage points. This tradeoff requires explicit governance documentation — organizations must decide acceptable thresholds for both metrics based on deployment risk profile.

**Recommended next steps:**
- Evaluate against PGD and Carlini-Wagner attacks
- Test FGSM robustness across epsilon values (0.01, 0.05, 0.1, 0.2)
- Investigate adversarial training mix ratios to optimize the robustness-utility balance

### Adversarial Visualization

Sample faces shown before and after FGSM perturbation alongside model predictions from both the baseline and adversarially trained models. Perturbation difference map highlights pixel-level changes introduced by the attack.

In [ ]:
with tf.device('/CPU:0'):
  num_samples_to_show = 5 # Number of samples to visualize

  plt.figure(figsize=(10, 2 * num_samples_to_show))

  # Get predictions from both models
  predictions_baseline_clean = (baseline.predict(X_test) > 0.5).astype(int)
  predictions_baseline_adv = (baseline.predict(X_test_adv) > 0.5).astype(int)
  predictions_adv_trained_clean = (adv_trained.predict(X_test) > 0.5).astype(int)
  predictions_adv_trained_adv = (adv_trained.predict(X_test_adv) > 0.5).astype(int)

  for i in range(num_samples_to_show):
      # Original Image
      plt.subplot(num_samples_to_show, 2, i * 2 + 1)
      plt.imshow(X_test[i].numpy().squeeze(), cmap='gray')
      plt.title(f"Original\nTrue: {y_test[i].numpy()}\nBaseline Pred: {predictions_baseline_clean[i][0]}\nAdvTrain Pred: {predictions_adv_trained_clean[i][0]}")
      plt.axis('off')

      # Adversarial Image
      plt.subplot(num_samples_to_show, 2, i * 2 + 2)
      plt.imshow(X_test_adv[i].numpy().squeeze(), cmap='gray')
      plt.title(f"Adversarial\nTrue: {y_test[i].numpy()}\nBaseline Pred: {predictions_baseline_adv[i][0]}\nAdvTrain Pred: {predictions_adv_trained_adv[i][0]}")
      plt.axis('off')

  plt.tight_layout()
  plt.show()

## Part D: Explainability & Transparency

### LIME Explainability

LIME (Local Interpretable Model-agnostic Explanations) applied to three test images to identify which facial regions drive gender classification decisions. 1,000 perturbed samples generated per explanation with top 5 superpixel features highlighted.

In [ ]:
# Define a prediction function for LIME
with tf.device('/CPU:0'):
  def predict_fn(images):
    # LIME passes numpy arrays.
    # If LIME feeds RGB (N,64,64,3), convert to (N,64,64,1) for our grayscale model
    if images.shape[-1] == 3:
        images = np.mean(images, axis=-1, keepdims=True)

    # Ensure float32 for TensorFlow model
    images = tf.convert_to_tensor(images, dtype=tf.float32)

    preds = baseline.predict(images, verbose=0)
    # LIME expects (N, num_classes) probabilities
    return np.hstack([1 - preds, preds])

  # Initialize LIME Image Explainer
  explainer = lime_image.LimeImageExplainer()

  # Choose a few images from the test set for explanation
  # Ensure these are numpy arrays and in the correct shape for LIME (height, width, channels)
  num_explanations = 3
  explanation_indices = [0, 1, 2] # Example indices

  plt.figure(figsize=(12, 4 * num_explanations))

  for i, idx in enumerate(explanation_indices):
      image_to_explain = X_test[idx].numpy() # Convert from torch.Tensor to numpy
      true_label = y_test[idx].numpy()

      # Generate explanation for the chosen image
      # num_features: number of superpixels to include in explanation
      # hide_color: color to use for hiding parts of the image
      explanation = explainer.explain_instance(
          image_to_explain.squeeze(), # LIME expects (H, W, C) so squeeze if it's (H, W, 1)
          predict_fn,
          top_labels=1,
          hide_color=0,
          num_samples=1000 # Number of perturbed samples to generate
      )

      # Get image and mask for the top predicted class
      temp, mask = explanation.get_image_and_mask(
          explanation.top_labels[0],
          positive_only=True,
          num_features=5,
          hide_rest=True
      )

      # Plot the original image and its LIME explanation
      plt.subplot(num_explanations, 2, i*2 + 1)
      plt.imshow(image_to_explain.squeeze(), cmap='gray')
      plt.title(f"Original Image (True: {true_label})")
      plt.axis('off')

      plt.subplot(num_explanations, 2, i*2 + 2)
      plt.imshow(mark_boundaries(temp / 2 + 0.5, mask))
      plt.title(f"LIME Explanation (Predicted: {explanation.top_labels[0]})")
      plt.axis('off')

  plt.tight_layout()
  plt.show()

In [ ]:
with tf.device('/CPU:0'):
  num_samples_to_show = 5 # Number of samples to visualize

  plt.figure(figsize=(15, 2 * num_samples_to_show))

  for i in range(num_samples_to_show):
      # Original Image
      plt.subplot(num_samples_to_show, 3, i * 3 + 1)
      plt.imshow(X_test[i].numpy().squeeze(), cmap='gray')
      plt.title(f"Original\nTrue: {y_test[i].numpy()}\nBaseline Pred: {predictions_baseline_clean[i][0]}\nAdvTrain Pred: {predictions_adv_trained_clean[i][0]}")
      plt.axis('off')

      # Adversarial Image
      plt.subplot(num_samples_to_show, 3, i * 3 + 2)
      plt.imshow(X_test_adv[i].numpy().squeeze(), cmap='gray')
      plt.title(f"Adversarial\nTrue: {y_test[i].numpy()}\nBaseline Pred: {predictions_baseline_adv[i][0]}\nAdvTrain Pred: {predictions_adv_trained_adv[i][0]}")
      plt.axis('off')

      # Difference Image (for visualization of perturbation)
      plt.subplot(num_samples_to_show, 3, i * 3 + 3)
      diff_image = np.abs(X_test[i].numpy().squeeze() - X_test_adv[i].numpy().squeeze())
      plt.imshow(diff_image, cmap='hot') # Using 'hot' colormap to highlight differences
      plt.title("Perturbation")
      plt.axis('off')

  plt.tight_layout()
  plt.show()

### Explainability Analysis

LIME heatmaps reveal the model primarily relies on facial edges and structural contours — nose, eyes, mouth, and jawline — rather than holistic facial features. The model successfully isolates facial regions from background, indicating it has learned spatially relevant patterns.

**Governance implication:** Explainability serves two functions in AI governance. When heatmaps highlight intuitive facial features, they build stakeholder confidence that the model is learning meaningful patterns rather than spurious correlations. When heatmaps highlight irrelevant regions, they flag potential bias requiring investigation before deployment.

The reliance on structural edge features rather than holistic facial understanding raises a governance concern — structural features correlate with demographic characteristics beyond gender, potentially introducing proxy discrimination. This warrants monitoring in any production deployment.

## Part E: Fairness & Bias Mitigation


#### Racial Group Performance Analysis Across Five Demographic Categories

Protected groups assessed across five racial categories (White, Black, Asian, Indian, Other) using TPR and FPR as primary fairness metrics. Two mitigation strategies applied and evaluated:

- **Re-weighting** — minority race groups assigned higher sample weights during training
- **Threshold calibration** — classification threshold lowered from 0.5 to 0.3 for the lowest-TPR demographic group

Fairness metrics compared before and after each mitigation strategy.

In [ ]:
with tf.device('/CPU:0'):
  # Part E: Fairness & Bias Mitigation
  # Identify Protected Group(s)
  # E.g., Race: 5 categories in UTKFace.

  race_name = {
      0: "White",
      1: "Black",
      2: "Asian",
      3: "Indian",
      4: "Other"
  }

  race_subset = df["race"].values[:len(X_tf)]

  race_train, race_test = train_test_split(
      race_subset,
      test_size = 0.2,
      stratify = y,
      random_state = 42
  )

  print("Race distribution test:")
  print(pd.Series(race_test).value_counts().sort_index().rename(index=race_name))

  from sklearn.metrics import precision_score, confusion_matrix

  def group_fairness_metrics(y_true, y_pred, group_arr):
      """
      Computes TPR (True Positive Rate) and FPR (False Positive Rate)
      for each group in the 'group_membership' array.
      """
      rows = []
      for g in sorted(np.unique(group_arr)):
          mask = (group_arr == g)
          yt = y_true[mask]
          yp = y_pred[mask]

          tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()

          acc = (tp + tn) / (tp + tn + fp + fn)

          tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
          fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
          # previously: fpr = fp / (fp + tn) if (tp + tn) > 0 else np.nan
          prec = precision_score(yt, yp, zero_division=0)

          rows.append({
              "group_id": int(g),
              "group": race_name.get(int(g), str(g)),
              "n": int(mask.sum()),
              "Accuracy": acc,
              "TPR": tpr,
              "FPR": fpr,
              "Precision": prec
            })
      return pd.DataFrame(rows).sort_values("group_id")

  def disparity(dfm, col):
    return float(dfm[col].max() - dfm[col].min())


  # Get predictions on the CPU
  with tf.device('/CPU:0'):
      y_pred = (baseline.predict(X_test) > 0.5).astype(int)
      y_true = y_test.numpy()


  # y_pred = (baseline.predict(X_test) > 0.5).astype(int)
  # y_true = y_test.numpy()

  fair_before = group_fairness_metrics(y_true, y_pred, race_test)

  print("Baseline fairness metrics:")
  print(fair_before)
  '''
  print("\nBaseline disparities:")
  print(f"Accuracy disparity:   {disparity(fair_before, 'Accuracy'):.4f}")
  print(f"\nTPR disparity:        {disparity(fair_before, 'TPR'):.4f}")
  print(f"\nFPR disparity:        {disparity(fair_before, 'FPR'):.4f}")
  print(f"\nPrecision disparity:  {disparity(fair_before, 'Precision'):.4f}")
  '''
  # Mitigation
  # Approach #1: Re-weight minority race examples

  race_counts = pd.Series(race_train).value_counts().to_dict()
  sample_weights = np.array([1.0 / race_counts[g] for g in race_train], dtype=np.float32)

  model_rw = build_model(X_train.shape[1:])
  compile_model(model_rw)

  model_rw.fit(
      X_train,
      y_train,
      sample_weight=sample_weights,
      epochs = 4,
      validation_data=(X_test, y_test)
  )

  # Approach #2: Post-processing threshold calibration for the group with lower TPR.

  y_prob = baseline.predict(X_test).flatten()
  lowest_group = fair_before.loc[fair_before["TPR"].idxmin(), "group_id"]
  print(f"\nLowest TPR group: {race_name.get(lowest_group, lowest_group)}")

  threshold_default = 0.5
  threshold_adjusted = 0.3

  y_pred_calibrated = []

  for i in range(len(y_prob)):
      if race_test[i] == lowest_group:
          pred = 1 if y_prob[i] > threshold_adjusted else 0
      else:
          pred = 1 if y_prob[i] > threshold_default else 0

      y_pred_calibrated.append(pred)

  y_pred_calibrated = np.array(y_pred_calibrated)

  # Re-evaluate
  # Compare group metrics before vs. after mitigation.

  fair_after_cal = group_fairness_metrics(y_true, y_pred_calibrated, race_test)

  print("Fairness metrics AFTER threshold calibration")
  display(fair_after_cal)

## Part F: Privacy-Enhancing Techniques


#### Differential Privacy and Federated Learning Evaluation

Two privacy-preserving training approaches evaluated against the baseline CNN:

- **Differential Privacy** — diffprivlib Logistic Regression trained across epsilon values (0.1, 0.5, 1.0, 5.0) to quantify the privacy-utility tradeoff
- **Federated Learning** — training data split across 3 simulated clients with naive weight averaging to demonstrate privacy-preserving distributed training

Results compared against baseline CNN accuracy of 70% to assess utility cost of each privacy mechanism.

In [ ]:
X_train_flat = X_train.numpy().reshape(X_train.shape[0], -1)
X_test_flat = X_test.numpy().reshape(X_test.shape[0], -1)

dp_model = DPLogisticRegression(epsilon=1.0, data_norm=2.0, max_iter=1000, random_state=42)
dp_model.fit(X_train_flat, y_train.numpy())
dp_preds = dp_model.predict(X_test_flat)
dp_acc = accuracy_score(y_test, dp_preds)

print("B) Differentially Private Logistic Regression:")
print(f"   Epsilon = 1.0")
print(f"   Accuracy = {dp_acc:.2f}\n")

epsilons = [0.1, 0.5, 1.0, 5.0]
dp_results = []

for eps in epsilons:

    dp_model = DPLogisticRegression(epsilon=eps, data_norm=2.0, max_iter=1000, random_state=42)
    dp_model.fit(X_train_flat, y_train.numpy())
    dp_preds = dp_model.predict(X_test_flat)
    dp_acc = accuracy_score(y_test, dp_preds)

    dp_results.append((eps, dp_acc))

dp_results_df = pd.DataFrame(dp_results, columns=["Epsilon", "Accuracy"])
display(dp_results_df)


In [ ]:
def split_into_clients(X, y, n_clients=3):
    X_clients = np.array_split(X, n_clients)
    y_clients = np.array_split(y, n_clients)
    return X_clients, y_clients

n_clients = 3
X_clients, y_clients = split_into_clients(X_train_flat, y_train.numpy(), n_clients=n_clients)

local_models = []
for i in range(n_clients):
    local_model = LogisticRegression(max_iter=1000, random_state=i)
    local_model.fit(X_clients[i], y_clients[i])
    local_models.append(local_model)

# Naive weight averaging (for illustration)
avg_coefs = np.mean([m.coef_ for m in local_models], axis=0)
avg_intercept = np.mean([m.intercept_ for m in local_models], axis=0)

# Perform a dummy fit with the entire training set to ensure at least 2 classes
fed_model = LogisticRegression(max_iter=1000, random_state=42)
fed_model.fit(X_train_flat, y_train.numpy())

# Overwrite the fitted coefficients with the aggregated (federated) ones
fed_model.coef_ = avg_coefs
fed_model.intercept_ = avg_intercept

fed_preds = fed_model.predict(X_test_flat)
fed_acc = accuracy_score(y_test, fed_preds)

with tf.device('/CPU:0'):
    loss_baseline, baseline_acc = baseline.evaluate(X_test, y_test, verbose=0)

print("D) Simulated Federated Learning with 3 local clients:")
print(f"   Accuracy after naive weight averaging = {fed_acc:.2f}\n")

print("SUMMARY OF RESULTS:")
print(f"   A) Baseline Model Accuracy:       {baseline_acc:.2f}")
print(f"   B) DP Model Accuracy (ε=1.0):     {dp_acc:.2f}")
print(f"   D) Federated Learning Accuracy:   {fed_acc:.2f}")


## Part G: Generative AI & Ethical Prompt Engineering


GPT-2 used to generate demographically diverse portrait prompts. Stable Diffusion v1.5 used to synthesize face-like images from generated prompts. Negative prompts applied to enforce ethical guidelines — preventing generation of harmful, stereotypical, or unrealistic outputs.

Demonstrates responsible generative AI deployment with explicit bias mitigation at the prompt engineering level.

In [ ]:
import logging
import gc
from transformers import logging as transformers_logging
# Suppress Warnings
transformers_logging.set_verbosity_error()

# Clear TensorFlow Session
tf.keras.backend.clear_session()
gc.collect()
print("TensorFlow session cleared.")

# Device Setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load GPT-2
model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

# Fix Padding
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Define Attributes
genders = ["male", "female"]
races = ["white", "black", "asian", "indian", "hispanic", "latino", "middle eastern"]
age_ranges = ["young", "middle-aged", "old"]
backgrounds = ["plain", "quite", "indoors", "busy", "outdoors"]
styles = ["casual", "professional", "candid", "headshot"]
expressions = ["happy", "sad", "blank", "frowning", "smiling"]
target_per_label = 120 # Reduced for demonstration speed

# Define sample_text Function
def sample_text(gender, race, age, background, style, expression, max_new_tokens=25):
  prompt = (
      f"Generate a photo with single person, focusing on their face with a {style} style of a {age} {race} {gender} showing a {expression} expression, set against a {background}-styled background."
  )
  # Tokenize and move to device
  inputs = tokenizer(prompt, return_tensors="pt")
  input_ids = inputs.input_ids.to(device)
  attention_mask = inputs.attention_mask.to(device)

  with torch.no_grad():
    out = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
  text = tokenizer.decode(out[0], skip_special_tokens=True)
  # Return only the generated suffix, not the original prompt
  return text.replace(prompt, "").strip()

# Generate Dataset
rows = []
print("Generating prompts...")
for gender in genders:
  count = 0
  while count < target_per_label:
    race = random.choice(races)
    age = random.choice(age_ranges)
    if age == "young":
      age_int = random.randint(18, 30)
    elif age == "middle-aged":
      age_int = random.randint(31, 60)
    else:
      age_int = random.randint(61, 90)

    background = random.choice(backgrounds)
    style = random.choice(styles)
    expression = random.choice(expressions)

    try:
        text = sample_text(gender, race, age, background, style, expression)
        if len(text.split()) >= 1:
          rows.append({
              "text": text,
              "gender": gender,
              "race": race,
              "age": age_int,
              "background": background,
              "style": style,
              "expression": expression
          })
          count += 1
    except Exception as e:
        print(f"Error generating prompt: {e}")
        continue

# Create DataFrame
df_outputs = pd.DataFrame(rows)

# Create Prompt Column
df_outputs["prompt_text"] = df_outputs.apply(
    lambda r: f"Generate a photo with single person, focusing on their face with a {r['style']} style of a {r['age']}-year old {r['race']} {r['gender']} showing a {r['expression']} expression, set against a {r['background']}-styled background. {r['text']}",
    axis=1
)

# Display Results
print("Dataset size:", len(df_outputs))
print(df_outputs.head())

In [ ]:
# Define the model ID for the Stable Diffusion model
sd_model = "runwayml/stable-diffusion-v1-5"

print(f"Loading Stable Diffusion model: {sd_model}...")
# Initialize the DiffusionPipeline
sd_pipeline = DiffusionPipeline.from_pretrained(sd_model, torch_dtype=device)

# Move the pipeline to the device
sd_pipeline.to(device)
print("Stable Diffusion pipeline initialized and moved to device.")

# Select a subset of prompts for image generation
# Ensure df_outputs is defined and contains 'prompt_text'
if 'df_outputs' in locals() and not df_outputs.empty:
    prompts_to_generate = df_outputs['prompt_text'].sample(n=5, random_state=SEED).tolist()
    print(f"Selected {len(prompts_to_generate)} prompts for image generation.")
    for i, prompt in enumerate(prompts_to_generate):
        print(f"Prompt {i+1}: {prompt}")
else:
    prompts_to_generate = [
        "A photo of a young male with brown hair, in a casual setting, looking forward.",
        "A natural portrait of an elderly female, smiling softly, with wrinkles around her eyes.",
        "A headshot of a middle-aged Asian person, serious expression, studio lighting.",
        "A candid shot of a Black person, early 20s, with short curly hair, outdoor background.",
        "A professional photo of an Indian female, 30s, in formal attire, neutral background."
    ]
    print("df_outputs not found or empty. Using default prompts for image generation.")
    for i, prompt in enumerate(prompts_to_generate):
        print(f"Default Prompt {i+1}: {prompt}")

# Generate images
print("Generating images...")
generated_images = []
for prompt in prompts_to_generate:
    # Ethical guideline: Avoid explicit negative prompts to encourage neutral, natural-looking portraits
    negative_prompt = "ugly, disfigured, deformed, bad anatomy, bad quality, blurry, cartoon, anime"
    image = sd_pipeline(prompt, negative_prompt=negative_prompt).images[0]
    generated_images.append(image)
    print(f"Generated image for prompt: '{prompt}'")

print("Image generation complete.")

# Visualize the generated images
plt.figure(figsize=(15, 5 * len(generated_images)))
for i, (image, prompt) in enumerate(zip(generated_images, prompts_to_generate)):
    plt.subplot(len(generated_images), 1, i + 1)
    plt.imshow(image)
    plt.title(f"Prompt: {prompt}", fontsize=10)
    plt.axis('off')
plt.tight_layout()
plt.show()

### Prompt Engineering Methodology

An adaptive prompt generation strategy constructs base prompts from predefined demographic attributes — gender, race, age, background, style, and expression — which GPT-2 then augments with natural language nuance. This ensures demographic diversity across generated portraits, aligning with the UTKFace dataset's demographic distribution.

**Negative prompts** served as an explicit ethical governance control, steering generation away from harmful outputs:

- Terms like "ugly, disfigured, deformed, bad anatomy" prevent generation of images that could be perceived as discriminatory or degrading
- Terms like "cartoon, anime" enforce realism — stylized faces can over-emphasize demographic features in ways that perpetuate stereotypes

This demonstrates a practical implementation of responsible AI deployment at the generation layer — governance applied not just to model training but to inference-time outputs.

## Conclusion & Discussion

### Key Trade-offs

**Utility vs. Robustness**

The baseline model achieved 70% accuracy on clean images but dropped to 14.5% under FGSM attack (epsilon=0.1) — a 55.5 percentage point degradation from imperceptible perturbations. Adversarial training recovered robustness to 100% on adversarial examples at a cost of 12.5 percentage points on clean accuracy (57.5%). Organizations must explicitly document acceptable thresholds for both metrics based on deployment risk profile.

**Utility vs. Privacy — Differential Privacy**

DP Logistic Regression showed accuracy ranging from 47-48% across epsilon values (0.1–5.0), compared to the 70% baseline CNN. The marginal accuracy difference across epsilon values suggests diminishing returns from tightening privacy constraints beyond epsilon=1.0 for this task and model architecture.

**Utility vs. Privacy — Federated Learning**

Simulated FL with 3 clients and naive weight averaging achieved 77% accuracy — outperforming both the baseline CNN (70%) and DP model (47%). FL offers a more favorable privacy-utility balance for this use case, though the Logistic Regression on flattened data is not directly comparable to the CNN architecture.

---

### Limitations

**Dataset constraints** — Processing limited to 1,000 images due to Colab free tier resource constraints. The full 23,708-image dataset would provide more representative fairness analysis, particularly for underrepresented racial groups.

**Model constraints** — 4 training epochs and CPU-only TensorFlow execution limited model performance. Production deployments would require significantly more training compute.

**Adversarial testing scope** — Analysis limited to FGSM at epsilon=0.1. Comprehensive robustness assessment requires PGD, Carlini-Wagner, and transfer attacks across multiple epsilon values.

**Mitigation simplicity** — Re-weighting, threshold calibration, basic DP, and naive FL are demonstration implementations. Production-grade bias mitigation and privacy preservation require more sophisticated approaches.